## Legal Doamin Example 
This example uses a [dataset of 60 verified GDPR formalization cases](https://huggingface.co/datasets/nguyenthanhasia/gdpr-cases) resulted from a verification-centered, human-in-the-loop framework for GDPR auto-formalization using role-specialized LLM agents. The interactive system can be used by legal students to understand and practice GDPR cases with varying premises. 

### 0. Import functions shared with the other domain examples

In [1]:
import sys
sys.path.append("..")
from yarrrmlHandler import YarrrmlHandler
from etc import export_sheets_to_csv
from rdf_etc import run_query_file, add_namespaces, show_rdf_graph

### 1. Export all Excel worksheets as csv files (YARRRML does not inherently support xlsx files)

In [2]:
for filename in ['claimGenerationNetwork', 'claimFormalizationNetwork']:
    export_sheets_to_csv(
        excel_file=filename + '.xlsx', 
        output_dir='csv'
)

Exported experts -> csv\experts.csv
Exported events -> csv\events.csv
Exported resources -> csv\resources.csv
Exported claims -> csv\claims.csv
Exported claimAssessments -> csv\claimAssessments.csv
Exported kbAssessments -> csv\kbAssessments.csv
Exported claims -> csv\claims.csv
Exported kEngineers -> csv\kEngineers.csv
Exported formalRepresentations -> csv\formalRepresentations.csv
Exported assessments -> csv\assessments.csv


### 2. Convert csv data to RDF representation using the YARRRML specification
The graph is visualized using [yfiles-graphs-for-jupter](https://www.yworks.com/products/yfiles-graphs-for-jupyter). Please refer to the official documentation for using the exploration features. The visualization is designed for networkx graphs, not RDF graphs in specific. If you want to explore the graph with a specialized tool, please export the graph by uncommenting the serialization contextmanager.

In [3]:
from rdflib import Graph

try:
    result = YarrrmlHandler.run_pipeline(
        working_dir=".",
        yarrrml_file="legal.yarrrml.yml",
        rml_file="legal_rules.rml.ttl",
    )
    print("Pipeline executed successfully.")
except Exception as ex:
    print(f"Error: {ex}")

g =  Graph().parse(data=result.stdout, format="nt")
g = add_namespaces(g)

# Uncomment to export RDF graph
# with open("rdf_export.ttl", mode="w", encoding="utf8") as f:
#     f.write(g.serialize(format="turtle"))

## Layout calculation takes a while for this case. Uncomment if desired.
# show_rdf_graph(g)

Pipeline executed successfully.


### 3. Store conditional knowledge base as annotation page as snapshot for further processing

In [4]:
from etc import output_pretty_rdf
        
scenario_rows = run_query_file(g, "get_all_scenarios.rq")

for row in scenario_rows:
    num = row['s'].split('_')[-1]
    result = run_query_file(g, "get_kb_template.rq", {"num": num})
    constructed_graph = Graph()
    for triple in result:
        constructed_graph.add(triple)
    constructed_graph = add_namespaces(constructed_graph)
    g += constructed_graph
    ttl = constructed_graph.serialize(format="turtle")

# Output kb of last scenario
output_pretty_rdf(ttl)

### 5. Define a weighting formula and a threshold to retrieve list of best fitting scenarios  

In [40]:
import pandas as pd

# Define a weighting formula to your needs.
FORMULA = "(?scenario_score + ?representation_score + ?logical_pass + ?legal_score) / 4"

# Change the threshold between 0.0 and 100.0
THRESHOLD = 91.5

passing_scenario_rows = run_query_file(g, "get_passing_scenarios.rq", {"FORMULA": FORMULA, "THRESHOLD": THRESHOLD})

df = pd.DataFrame(
    [
        (
            int(str(row.scenario).split("_")[1]),
            str(row.calculated_score),
            str(row.scenario_score),
            str(row.representation_score),
            str(row.logical_pass),
            str(row.legal_score)
        )
        for row in passing_scenario_rows
    ],
    columns=["Scenario Number", "Calculated Score", "Scenario Score", "Representation Score", "Logical Pass", "Legal Score"]  
).set_index("Scenario Number")


display(df)

,Calculated Score,Scenario Score,Representation Score,Logical Pass,Legal Score
Scenario Number,,,,,
318,93.5,92.0,100,100,82
383,93.5,92.0,100,100,82
305,93.0,87.0,100,100,85
367,92.75,93.0,100,100,78
148,92.5,92.0,100,100,78
241,92.5,92.0,100,100,78
299,92.5,87.0,100,100,83
362,92.5,92.0,100,100,78
389,92.5,92.0,100,100,78


### 6. Generate the Evaluation system based on a selected scenario
* Assign a valid scenario id for SCENARIO_NUM and execute block.
* Select facts and propositions to be true for the reasoning the _Facts tab_.
* Choose a target proposition in the _Target predicates tab_ and press "Evaluate" to start the reasoning process.

In [41]:
from IPython.display import display
from pythen_etc import collect_predicates, compose_scenario
from gui import create_gui

from pythen import RuleTreeEvaluator, RuleValidationError

# Choose scenario number from step 5
SCENARIO_NUM = 318

try:
    scenario = compose_scenario(g, SCENARIO_NUM)
    evaluator = RuleTreeEvaluator(scenario["rule_tree"], enable_trace=True)
    predicates = collect_predicates(evaluator.rule_tree)
    gui = create_gui(predicates, scenario["facts"], evaluator)
    display(gui["tabs"])
except RuleValidationError as e:
    print(f"Scenario {SCENARIO_NUM} does not exist. Check the claims spreadsheet for available scenario ids.")

2026-09-27 13:11:31,717 - INFO - RuleTreeEvaluator initialized with 7 rules
